# BB scattering analysis: Part 2

In this notebook, we will take a second pass at the analysis for your BB scattering data. This time however, we will consider the uncertainty in the data and see how that affects our analysis technique and, ultimately, our results. 

Let's reset your analysis in the cell below. Import numpy and matplotlib. Reassign the values you had in your last notebook (`dx`, `L`, `num_events_tot`, etc.). Plot your data to make sure that there are no errors in your script. For simplicity, everyone should use the `N` as a function of `sin(theta/2)` figure for today's analysis.

In [ ]:
# Your code goes here.

## Y-errorbars

The y-axis is counting the number of events. For counting experiments, the uncertainty on $N$ events is $\sqrt{N}$. 

In the cell below, create an array of values for the uncertainty in `num_events_tot`. Label that array `d_num_events_tot`. Then plot those results using the `errorbar` function.

In [ ]:
d_num_events_tot = ?

fig = plt.figure()
ax = fig.add_subplot(111)
ax.errorbar(?, ?, yerr=d_num_events_tot, fmt='o', capsize=5)

## X-errorbars

The x-axis is $\sin(\theta/2)$. We need to find the uncertainty in that value. The experimental values that went into the x-axis are:
- `r_detector`
- `dx`

To get a sense of the size of the error on those values, you would have to do repeated measurements to get a mean and standard deviation. If you made the initial measurement carefully, that's likely a relatively small error when compared to the size of the error along the y-axis. There might be additional errors in the measured angle due to how the tape was mounted and the location of $\theta=0$. Here, we are left to make an educated guess. For this analysis, we'll assume that the angle is precise within $\pm1$ degree, or $\pm$0.0174 rad.

You need to use the partial derivative method to calculate $\Delta\sin(\theta/2)$ from $\Delta\theta$. Do that calculation and assign the value to `d_sin_theta`. Then plot the data including both xerr and yerr.

In [ ]:
# Your code goes here

## Weighted fits

The errorbars on a data point are the measure of confidence that repeating the experiment would yield the same data point. This can come into play when we fit our data to know models to verify agreement, look for new trends, etc. You can force the fit to give more significance to data points with smaller errorbars by using weighted fit algorithms. 

To learn how to do this, imagine that you ran a free fall experiment. The data is written in the cell below:

In [ ]:
time = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0])
position = np.array([0.011, 0.046, 0.094, 0.172, 0.290, 0.429, 0.526, 0.727, 0.811, 1.129, 1.232, 1.444, 1.841, 2.040, 2.529, 3.005, 3.180, 3.639, 3.714, 4.405])
d_position = np.array([0.002, 0.008, 0.018, 0.032, 0.050, 0.072, 0.098, 0.128, 0.162, 0.200, 0.242, 0.288, 0.338, 0.392, 0.450, 0.512, 0.578, 0.648, 0.722, 0.800])
d_time = np.array([0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03, 0.03])

fig = plt.figure()
ax = fig.add_subplot(111)
ax.errorbar(time, position, xerr=d_time, yerr=d_position, fmt='o', capsize=5)

To create the weighted fit, we use the Orthogonal Distance Regression method in scipy as shown below. The fit function has a different format than before. Now, its first argument is an array of fit parameters. The second argument is the independent variable.

In [ ]:
# Fitting function
def freefall(beta, t):
    # beta contains: initial position, initial velocity, and the acceleration
    initP, initV, acc = beta 
    return initP + initV*t + 0.5*acc*pow(t, 2)
    
# Initial guesses of the fit parameters. Make sure that the order matches that of the function definition for the model.
# [initP, initV, acc]
init_vals = [0., 0., 2.]

# Model object
model = odr.Model(freefall)

# Create RealData object
data = odr.RealData(time, position, sx=d_time, sy=d_position)

# Setup odr with model and data
myodr = odr.ODR(data, model, beta0=init_vals)

# Run the regression
out = myodr.run()

popt = out.beta
errorbars = out.sd_beta
init_position = popt[0]
d_init_position = errorbars[0]
init_velocity = popt[1]
d_init_velocity = errorbars[1]
acc = popt[2]
d_acc = errorbars[2]

# Print fit parameters
print(f'The fit parameters are: \n init_position = {init_position:.3f} +/- {d_init_position:.3f} \n init_velocity = {init_velocity:.3f} +/- {d_init_velocity:.3f} \n a = {acc:.3f} +/- {d_acc}')

time_long = np.linspace(time[0], time[-1], 100)
fig = plt.figure()
ax = fig.add_subplot(111)
ax.errorbar(time, position, xerr=d_time, yerr=d_position, fmt='o', capsize=5)
ax.plot(time_long, freefall(popt, time_long), color='r')

Now repeat those steps for your BB scattering experiment.

In [ ]:
# Your code goes here